# IPLMS Forecast — data gate và offline experiment

Notebook này chỉ điều phối code trong `training/forecast/`. Nó không chứa business logic, không promote model và không bật runtime Forecast. Dữ liệu synthetic chỉ dùng smoke test, không dùng để công bố chất lượng.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import subprocess
import sys

DATA_LAKE = Path("data/datalake")
AS_OF = "2026-09-25T00:00:00+07:00"  # Giữ cố định cho cùng một experiment.
datetime.fromisoformat(AS_OF)  # Fail sớm nếu timestamp không hợp lệ.

## 1. Kiểm tra data contract

Gate mặc định cần manifest/checksum hợp lệ, watermark incremental và ít nhất một bãi có 30 ngày lịch sử.

In [ ]:
from training.forecast.run_experiment import summarize_readiness
from training.readiness import inspect_forecast

readiness = inspect_forecast(DATA_LAKE, min_history_days=30)
print(json.dumps(summarize_readiness(readiness), ensure_ascii=False, indent=2))

## 2. Chạy pipeline bất biến

Khi gate đạt, lệnh tạo snapshot → baseline → point-in-time features → ridge candidate → một run report liên kết toàn bộ artifact. Exit code `1` nghĩa là candidate không thắng persistence; artifact vẫn được giữ để audit.

In [ ]:
if not readiness["ready"]:
    print("BLOCKED: bổ sung lịch sử thật trước khi train; không hạ gate để lấy metric đẹp.")
    experiment = None
else:
    command = [
        sys.executable, "-m", "training.forecast.run_experiment",
        "--as-of", AS_OF,
        "--data-lake", str(DATA_LAKE),
    ]
    experiment = subprocess.run(command, text=True, capture_output=True)
    print(experiment.stdout)
    if experiment.stderr:
        print(experiment.stderr)

## 3. Đọc run report

Chỉ so sánh các run có cùng định nghĩa target, horizon và cửa sổ dữ liệu. Không đưa candidate vào runtime trước bước review + shadow rollout.

In [ ]:
reports = sorted((DATA_LAKE / "reports" / "forecast-runs").glob("*.json"))
if not reports:
    print("Chưa có run report hợp lệ.")
else:
    report = json.loads(reports[-1].read_text(encoding="utf-8"))
    print("run_id:", report["run_id"])
    print("status:", report["status"])
    print("lots:", len(report["eligible_lots"]))
    print("validation metrics:")
    print(json.dumps(report["validation_metrics"], indent=2))